# HuggingFace client

#### 1. Dependencies

In [ ]:
%pip install --quiet openai pandas tqdm httpx

In [ ]:
import os
import time
import httpx
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

#### 2. Configuration

In [ ]:
MODEL_NAME = "CYFRAGOVPL/PLLuM-12B-instruct"
MODEL_SHORT = "PLLuM-12B-HF"

LANG_FILTER = "pl"

ENDPOINT_URL = ""

def load_api_key(path, name):
    for line in open(path).read().strip().splitlines():
        if line.startswith(f"{name}="):
            return line.split("=", 1)[1].strip()
    raise KeyError(f"Klucz '{name}' nie znaleziony w {path}")

API_KEY = load_api_key("./api_key.txt", "huggingface")

if not ENDPOINT_URL:
    raise ValueError("Ustaw ENDPOINT_URL - wdroz endpoint na https://endpoints.huggingface.co i wklej URL")
if not LANG_FILTER:
    raise ValueError("Ustaw LANG_FILTER - modele HF-only sa monolingualne")

PROMPTS_PATH = "./prompts.tsv"
OUTPUT_DIR = f"./responses_{MODEL_SHORT}"
RESPONSES_PATH = f"{OUTPUT_DIR}/responses.tsv"

POLITE_DELAY = 1.0
TIMEOUT = 555.0
MAX_OUTPUT_TOKENS = 8192

TARGET_LANGUAGES = ["zh", "fi", "fr", "he", "ja", "pl"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

#### 3. Load prompts

In [ ]:
prompts_df = pd.read_csv(PROMPTS_PATH, sep="\t")

if LANG_FILTER is not None:
    langs = [LANG_FILTER] if isinstance(LANG_FILTER, str) else list(LANG_FILTER)
    prompts_df = prompts_df[prompts_df["target_lang"].isin(langs)].reset_index(drop=True)
    print(f"Filtr jezyka: {langs} -> {len(prompts_df)} promptow")

try:
    df = pd.read_csv(RESPONSES_PATH, sep="\t")
    df.loc[df["response"].astype(str).str.startswith("EXCEPTION:"), "response"] = pd.NA
    done = df["response"].notna().sum()
    print(f"Wznowiono: {done}/{len(df)} wykonanych")
except FileNotFoundError:
    df = prompts_df.copy()
    df["response"] = pd.NA
    print(f"Start od poczatku: {len(df)} promptow do wyslania")

#### 4. Send loop

In [ ]:
client = OpenAI(
    base_url=ENDPOINT_URL,
    api_key=API_KEY,
    http_client=httpx.Client(timeout=httpx.Timeout(TIMEOUT)),
)

MAX_RETRIES = 4
RETRY_BASE_DELAY = 15.0

def is_transient(exc):
    msg = str(exc).upper()
    return any(kw in msg for kw in ("429", "503", "502", "504", "RATE LIMIT", "OVERLOAD", "TRY AGAIN", "UNAVAILABLE", "TIMEOUT", "TEMPORARILY"))

def send_one(prompt):
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model="tgi",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=MAX_OUTPUT_TOKENS,
            )
            return response.choices[0].message.content or ""
        except Exception as exc:
            if is_transient(exc) and attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_BASE_DELAY * (2 ** attempt))
            else:
                raise

pending = df[df["response"].isna()].index.tolist()
print(f"Do wyslania: {len(pending)} (delay={POLITE_DELAY}s)")

failed = 0
for i in tqdm(pending, desc="Wysylanie"):
    time.sleep(POLITE_DELAY)
    try:
        df.at[i, "response"] = send_one(df.at[i, "prompt"])
    except Exception:
        df.at[i, "response"] = pd.NA
        failed += 1
    df.to_csv(RESPONSES_PATH, sep="\t", index=False)

print(f"Sukces: {len(pending) - failed}, blad: {failed}")
print("Pamietaj usunac endpoint na huggingface.co!")

#### 5. Per-language split

In [ ]:
for lang in TARGET_LANGUAGES:
    mask = df["id"].str.startswith(f"{lang}--")
    if not mask.any():
        continue
    df_lang = df[mask].copy()
    df_lang["id"] = df_lang["id"].str.replace(f"^{lang}--", "", regex=True)
    out_path = f"{OUTPUT_DIR}/{MODEL_SHORT}__{lang}__original.csv"
    df_lang.to_csv(out_path, index=False)
    print(f"  {lang}: {len(df_lang)} wierszy -> {out_path}")

#### 6. Display sample

In [ ]:
print(df.iloc[0]["response"][:1500] if pd.notna(df.iloc[0]["response"]) else "brak odpowiedzi")